# SO-101 Perception Benchmark 3 — multi-object, per-object, per-axis

20 scenes of {red cube, green sphere, blue **tall** box, yellow cylinder} + grey container. Each object is found by its **own SAM 3 text prompt** (no coordinates leaked), localised two ways, and scored **per axis** vs held-out ground truth:
- **floor_shape** — SAM 3 mask → shape-aware geometry (floor contact for x,y; height inferred from the silhouette per shape). No depth model.
- **depthpro** — SAM 3 centroid → Depth Pro depth (floor-anchored) → deproject (baseline).

Run §0, install §1 and **Restart**, then run the rest. SAM 3 needs `facebook/sam3` access + an `HF_TOKEN` secret. Last cell pushes the log.

## 0. Setup + logger

In [ ]:
import os, sys
if not os.path.exists('camera_math.py'):
    os.system('git clone https://github.com/Yunsmn/RoboticsPerceptionTest.git')
    os.chdir('RoboticsPerceptionTest')
sys.path.insert(0, '.')
import numpy as np, json
from pathlib import Path
from PIL import Image
import camera_math as CM
print('setup OK')

In [ ]:
import sys, io, datetime, traceback, subprocess
from IPython import get_ipython
_LOG = 'run_log.md'; _f = open(_LOG, 'w')
def _w(s=''):
    _f.write(str(s) + '\n'); _f.flush()
_w('# Run log (benchmark3) — ' + datetime.datetime.now().isoformat(timespec='seconds'))
try:
    _gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                          capture_output=True, text=True).stdout.strip()
except Exception:
    _gpu = ''
_w('- GPU: ' + (_gpu or 'none / CPU')); _w('- Python: ' + sys.version.split()[0])
class _Tee:
    _is_tee = True
    def __init__(self, real): self._real = real
    def write(self, s):
        n = self._real.write(s)
        try: _f.write(s); _f.flush()
        except Exception: pass
        return n
    def flush(self):
        try: self._real.flush()
        except Exception: pass
    def __getattr__(self, k):
        return getattr(self.__dict__['_real'], k)
if not getattr(sys.stdout, '_is_tee', False): sys.stdout = _Tee(sys.stdout)
if not getattr(sys.stderr, '_is_tee', False): sys.stderr = _Tee(sys.stderr)
_ip = get_ipython(); _n = {'i': 0}
def _pre(info):
    _n['i'] += 1
    _w(); _w('## Cell ' + str(_n['i'])); _w('```python')
    _w((info.raw_cell or '').rstrip()); _w('```'); _w('**output:**'); _w('```text')
def _post(res):
    _w('```')
    err = getattr(res,'error_in_exec',None) or getattr(res,'error_before_exec',None)
    if err is not None:
        _w('**ERROR:**'); _w('```text')
        _w(''.join(traceback.format_exception(type(err), err, err.__traceback__))); _w('```')
if not globals().get('_LOGGER_ON'):
    _ip.events.register('pre_run_cell', _pre); _ip.events.register('post_run_cell', _post); _LOGGER_ON = True
print('run logger active -> run_log.md')

## 1. Install (Depth Pro + SAM 3) — then Runtime → Restart

In [ ]:
# Depth Pro + SAM 3 deps. RESTART after (numpy ABI). sam3.pt is fetched post-restart in the
# adapter (using the HF_TOKEN Colab secret) so its output is logged and path-independent.
get_ipython().system('pip -q install git+https://github.com/apple/ml-depth-pro.git')
get_ipython().system('pip -q install ultralytics')
get_ipython().system('pip -q install -U git+https://github.com/ultralytics/CLIP.git')   # SAM3 tokenizer fix

## 2. Data + floor anchor + shape factor

In [ ]:
man = json.loads(Path('data_multi/manifest.json').read_text()); cam = man['camera']
f, cx, cy = cam['f'], cam['cx'], cam['cy']
cam_pos = np.array(cam['cam_pos'], float); R_cw = np.array(cam['R_cw'], float)
FRAMES = man['frames']
KIND = {o['name']: o['kind'] for o in man['objects']}      # from the prompt/identity, not GT geometry
PROMPT = {o['name']: o['prompt'] for o in man['objects']}
print(len(FRAMES), 'scenes |', {n: PROMPT[n] for n in PROMPT})

def project(P):
    xc, yc, zc = R_cw.T @ (np.asarray(P, float) - cam_pos)
    return cx + f * xc / -zc, cy - f * yc / -zc, -zc

# known floor pixels (z=0) for the Depth Pro scale anchor
FLOOR_PTS = []
for vv in range(300, 470, 10):
    for uu in range(40, 600, 20):
        hit = CM.ray_plane(uu, vv, f, cx, cy, cam_pos, R_cw, 0.0)
        if hit is None: continue
        dt = project([hit[0], hit[1], 0.0])[2]
        if 0.5 < dt < 3.0: FLOOR_PTS.append((int(uu), int(vv), float(dt)))
FU = np.array([p[0] for p in FLOOR_PTS]); FV = np.array([p[1] for p in FLOOR_PTS]); FT = np.array([p[2] for p in FLOOR_PTS])

def box_factor(x, y, kind):
    # silhouette Δv / (f*h/D) for a reference shape at (x,y). Size-independent; kind-specific.
    ref = {'box': (0.015, 0.015, 0.015), 'box_tall': (0.015, 0.015, 0.015), 'cylinder': (0.015, 0.025)}[kind]
    if kind == 'cylinder':
        r, hh = ref; pts = [np.array([x, y, hh]) + [r*np.cos(p), r*np.sin(p), sz]
                            for sz in (-hh, hh) for p in np.linspace(0, 2*np.pi, 24)]
    else:
        hx, hy, hz = ref; hh = hz; pts = [np.array([x, y, hz]) + [sx*hx, sy*hy, sz*hz]
                            for sx in (-1,1) for sy in (-1,1) for sz in (-1,1)]
    uv = np.array([project(P)[:2] for P in pts])
    return (uv[:,1].max() - uv[:,1].min()) / (f * (2*hh) / project([x, y, 0.0])[2])

## 3. Adapters (Depth Pro + SAM 3 features)

In [ ]:
import torch
_DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', _DEV); _M = {}

def _dp():
    import depth_pro, os
    if 'dp' not in _M:
        ckpt = 'checkpoints/depth_pro.pt'
        if not (os.path.exists(ckpt) and os.path.getsize(ckpt) > 10_000_000):
            os.makedirs('checkpoints', exist_ok=True)
            print('downloading Depth Pro checkpoint (~1.9GB)...')
            os.system('wget -q https://ml-site.cdn-apple.com/models/depth-pro/depth_pro.pt -O ' + ckpt)
        m, tf = depth_pro.create_model_and_transforms(device=torch.device(_DEV)); _M['dp'] = (m.eval(), tf)
    return _M['dp']

def depth_map(img):
    m, tf = _dp()
    with torch.no_grad():
        pred = m.infer(tf(img), f_px=torch.tensor(float(f), device=_DEV))
    return pred['depth'].squeeze().float().cpu().numpy()

_pred = None
def _sam3():
    global _pred
    if _pred is None:
        from huggingface_hub import hf_hub_download
        from google.colab import userdata
        from ultralytics.models.sam import SAM3SemanticPredictor
        try: tok = userdata.get('HF_TOKEN')
        except Exception: tok = None
        if not tok:
            raise RuntimeError("HF_TOKEN Colab secret missing/empty -> add it (key icon) with Notebook "
                               "access ON; SAM 3 (facebook/sam3) is gated.")
        pt = hf_hub_download('facebook/sam3', 'sam3.pt', token=tok)   # returns the cached path (no cwd dep)
        print('sam3.pt ->', pt)
        _pred = SAM3SemanticPredictor(overrides=dict(conf=0.25, task='segment', mode='predict',
                                      model=pt, save=False, verbose=False))
    return _pred

def sam3_features(img_rgb, prompt):
    # SAM 3 open-vocab TEXT prompt -> highest-confidence mask -> silhouette features (no GT).
    _pred = _sam3()
    _pred.set_image(img_rgb)
    r = _pred(text=[prompt])[0]
    if getattr(r, 'masks', None) is None or len(r.masks) == 0:
        return None
    m = r.masks.data.cpu().numpy()
    conf = r.boxes.conf.cpu().numpy() if getattr(r, 'boxes', None) is not None else np.ones(len(m))
    mask = m[int(np.argmax(conf))] > 0.5
    ys, xs = np.where(mask)
    if len(xs) == 0: return None
    vb = int(ys.max()); ubot = float(xs[ys >= vb - 2].mean())
    return {'cen': (float(xs.mean()), float(ys.mean())), 'vt': int(ys.min()), 'vb': vb,
            'umin': int(xs.min()), 'umax': int(xs.max()), 'ubot': ubot}

## 4. Localisation methods

In [ ]:
# localise a detected object -> (x,y,z) in the base frame. kind (from the prompt) picks the geometry.
def loc_floor(kind, fe):
    hit0 = CM.ray_plane(fe['ubot'], fe['vb'], f, cx, cy, cam_pos, R_cw, 0.0)   # bottom contact on floor
    if hit0 is None: return None
    x0, y0 = hit0; D0 = project([x0, y0, 0.0])[2]
    if kind == 'sphere':
        zc = (fe['umax'] - fe['umin']) * D0 / f / 2.0            # radius from mask width
    elif kind == 'container':
        return np.array([x0, y0, 0.0])                           # container: (x,y) on the floor
    else:                                                        # box / box_tall / cylinder
        zc = ((fe['vb'] - fe['vt']) * D0 / f / box_factor(x0, y0, kind)) / 2.0
    hit = CM.ray_plane(fe['cen'][0], fe['cen'][1], f, cx, cy, cam_pos, R_cw, zc)
    return None if hit is None else np.array([hit[0], hit[1], zc])

def loc_depthpro(fe, dm):
    fs = float(np.median(FT / dm[FV, FU]))                       # floor-anchor scale
    u, v = fe['cen']; d = float(dm[int(round(v)), int(round(u))]) * fs
    return CM.point_at_depth(u, v, f, cx, cy, cam_pos, R_cw, d)

## 5. Evaluate (per object, per axis)

In [ ]:
def perax(P, G): return np.abs(np.asarray(P, float) - np.asarray(G, float)) * 1000

names = list(KIND)
err = {m: {n: [] for n in names} for m in ('floor_shape', 'depthpro')}
detpx = {n: [] for n in names}; miss = {n: 0 for n in names}
for fr in FRAMES:
    img = np.array(Image.open('data_multi/' + fr['png']).convert('RGB'))
    dm = None
    for n in names:
        fe = None
        try: fe = sam3_features(img, PROMPT[n])
        except Exception as e:
            if sum(miss.values()) == 0: print('sam3 error:', e)
        if fe is None: miss[n] += 1; continue
        g = np.array(fr['objects'][n]['gt_xyz'])
        detpx[n].append(np.hypot(fe['cen'][0]-fr['objects'][n]['gt_uv'][0], fe['cen'][1]-fr['objects'][n]['gt_uv'][1]))
        Pf = loc_floor(KIND[n], fe)
        if Pf is not None: err['floor_shape'][n].append(perax(Pf, g))
        try:
            if dm is None: dm = depth_map(img)
            err['depthpro'][n].append(perax(loc_depthpro(fe, dm), g))
        except Exception as e:
            if sum(len(v) for v in err['depthpro'].values()) == 0: print('depthpro error:', e)

def summ(lst):
    if not lst: return None
    E = np.array(lst); loc = np.linalg.norm(E, axis=1)
    return {'n': len(E), 'x': round(float(np.median(E[:,0])),1), 'y': round(float(np.median(E[:,1])),1),
            'z': round(float(np.median(E[:,2])),1), 'loc_med': round(float(np.median(loc)),1),
            'loc_p95': round(float(np.percentile(loc,95)),1)}
results = {m: {n: summ(err[m][n]) for n in names} for m in err}
print('SAM 3 detection: ', {n: 'det %d/%d, px %.1f' % (len(detpx[n]), len(FRAMES),
      (np.median(detpx[n]) if detpx[n] else -1)) for n in names})

## 6. Compare

In [ ]:
print('\\n=== benchmark 3: per-object per-axis error (mm), base frame ===')
for m in ('floor_shape', 'depthpro'):
    print('\\n[%s]  (x / y / z / loc-med / loc-p95)' % m)
    for n in KIND:
        r = results[m][n]
        if r: print('  %-13s (%-9s) x %5.1f  y %5.1f  z %5.1f  | loc %5.1f / %-5.1f  (n=%d)'
                    % (n, KIND[n], r['x'], r['y'], r['z'], r['loc_med'], r['loc_p95'], r['n']))
        else: print('  %-13s -> no detections' % n)
print('\\nlocal ceilings (perfect masks): cube 0.5 sphere 0.7 cyl 0.7 | tall box ~18 (aspect ambiguity).')
print('bar: ~2mm grasp. Container: x,y only (cm-tolerant).')

## 7. Push the run log

In [ ]:
import subprocess, datetime, os, shutil
from google.colab import userdata
print('=== results snapshot ===')
print('results:', globals().get('results', '(eval cell not run yet)'))
def _sh(args, secret=None):
    r = subprocess.run(args, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if secret and out: out = out.replace(secret, '***')
    if out: print(out)
    return r.returncode
try: _TOKEN = userdata.get('GH_TOKEN')
except Exception as _e: _TOKEN = None; print('No GH_TOKEN secret:', _e)
if not _TOKEN:
    print('Set GH_TOKEN in the Colab Secrets panel, then re-run.')
else:
    _REPO = 'Yunsmn/RoboticsPerceptionTest'
    try: _f.flush()
    except Exception: pass
    os.makedirs('logs', exist_ok=True)
    _stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S'); _dest = 'logs/run3_%s.md' % _stamp
    shutil.copy('run_log.md', _dest)
    _url = 'https://%s@github.com/%s.git' % (_TOKEN, _REPO)
    _sh(['git','config','user.email','younesosf@gmail.com']); _sh(['git','config','user.name','Yunsmn'])
    _sh(['git','pull','--rebase','--autostash', _url, 'main'], secret=_TOKEN)
    _sh(['git','add', _dest]); _sh(['git','add', 'viz'])
    _sh(['git','commit','-m','log: benchmark3 run %s' % _stamp])
    _rc = _sh(['git','push', _url, 'HEAD:main'], secret=_TOKEN)
    print(('PUSHED ' if _rc == 0 else 'PUSH FAILED (rc=%d) ' % _rc) + _dest)
    print('-> tell the author: pull and read ' + _dest)